# 08. Data Sources, File Formats & Partitioning on Disk

In enterprise data pipelines, choosing the right file format and partitioning strategy determines whether your queries take minutes or hours.

### Topics Covered:
1. **Row-based (CSV, JSON) vs Columnar (Parquet, ORC) Storage**
2. **Reading & Writing CSV with schemas and options**
3. **Reading & Writing JSON**
4. **Reading & Writing Parquet (The big data gold standard)**
5. **SaveModes: `overwrite`, `append`, `ignore`, `errorIfExists`**
6. **Data Partitioning on Disk (`partitionBy`) and Partition Pruning**
7. **The Small Files Problem (and how to avoid it)**
8. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [1]:


import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

spark = SparkSession.builder \
    .appName("08_Data_Sources") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession initialized!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 18:55:36 WARN Utils: Your hostname, Tiger, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 18:55:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 18:55:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialized!


---
## 2. File Formats: Row-Based vs Columnar

| Format | Storage Style | Compression | Best For | Downsides |
| :--- | :--- | :--- | :--- | :--- |
| **CSV** | Row-based (plain text) | Poor | Human readability, quick export | No schema, slow parsing, large size |
| **JSON** | Row-based (semi-structured) | Fair | Web APIs, nested documents | Bulky keys repeated on every row |
| **Parquet** | **Columnar** (binary) | **Excellent (Snappy)** | **Data Warehousing, Analytics, Big Data** | Not human-readable in Notepad |

### Why is Parquet the Big Data Standard?
1. **Column Pruning**: If your table has 100 columns and your query only selects 3 columns, Parquet reads ONLY those 3 columns from disk. CSV must read the entire file!
2. **Predicate Pushdown**: Parquet files store statistics (min/max values) for each data chunk. Spark skips entire files without even opening them if the searched value is outside the min/max range!
3. **High Compression**: Snappy compression reduces storage size by 70%-90%.


---
## 3. Creating Sample Data to Test I/O


In [2]:
# Sample e-commerce data
orders_data = [
    (1, "Alice", "US", "Electronics", 999.99, 2023),
    (2, "Bob", "UK", "Clothing", 49.99, 2023),
    (3, "Charlie", "US", "Electronics", 120.00, 2023),
    (4, "Diana", "CA", "Books", 15.50, 2022),
    (5, "Evan", "US", "Clothing", 75.00, 2022),
    (6, "Fiona", "UK", "Electronics", 450.00, 2023),
    (7, "George", "CA", "Clothing", 110.00, 2023),
    (8, "Hannah", "US", "Books", 35.00, 2022)
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer", StringType(), True),
    StructField("country", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("year", IntegerType(), True)
])

df = spark.createDataFrame(orders_data, schema=orders_schema)
df.show()


+--------+--------+-------+-----------+------+----+
|order_id|customer|country|   category|amount|year|
+--------+--------+-------+-----------+------+----+
|       1|   Alice|     US|Electronics|999.99|2023|
|       2|     Bob|     UK|   Clothing| 49.99|2023|
|       3| Charlie|     US|Electronics| 120.0|2023|
|       4|   Diana|     CA|      Books|  15.5|2022|
|       5|    Evan|     US|   Clothing|  75.0|2022|
|       6|   Fiona|     UK|Electronics| 450.0|2023|
|       7|  George|     CA|   Clothing| 110.0|2023|
|       8|  Hannah|     US|      Books|  35.0|2022|
+--------+--------+-------+-----------+------+----+



---
## 4. Reading and Writing CSV Files

Key options when working with CSV:
* `header=True`: First row contains column names.
* `inferSchema=True`: Automatically detects types (or better: pass explicit schema).
* `sep=","`: Delimiter character (can be comma, tab `\t`, pipe `|`, etc.).


In [3]:
# Temp folder for testing
tmp_dir = "temp_spark_data"
csv_path = os.path.join(tmp_dir, "csv_output")

# Write to CSV with header
df.write.mode("overwrite").option("header", True).csv(csv_path)
print(f"Data written to CSV at: {csv_path}")

# Read CSV back with explicit schema
read_csv_df = spark.read \
    .option("header", True) \
    .schema(orders_schema) \
    .csv(csv_path)

print("Read back from CSV:")
read_csv_df.show(3)


Data written to CSV at: temp_spark_data/csv_output
Read back from CSV:
+--------+--------+-------+-----------+------+----+
|order_id|customer|country|   category|amount|year|
+--------+--------+-------+-----------+------+----+
|       3| Charlie|     US|Electronics| 120.0|2023|
|       1|   Alice|     US|Electronics|999.99|2023|
|       6|   Fiona|     UK|Electronics| 450.0|2023|
+--------+--------+-------+-----------+------+----+
only showing top 3 rows


---
## 5. Reading and Writing JSON Files


In [4]:
json_path = os.path.join(tmp_dir, "json_output")

# Write to JSON
df.write.mode("overwrite").json(json_path)

# Read from JSON
read_json_df = spark.read.json(json_path)
print("Read back from JSON:")
read_json_df.show(3)


Read back from JSON:
+------+-----------+-------+--------+--------+----+
|amount|   category|country|customer|order_id|year|
+------+-----------+-------+--------+--------+----+
| 120.0|Electronics|     US| Charlie|       3|2023|
|999.99|Electronics|     US|   Alice|       1|2023|
| 450.0|Electronics|     UK|   Fiona|       6|2023|
+------+-----------+-------+--------+--------+----+
only showing top 3 rows


---
## 6. Reading and Writing Parquet (The Production Standard)

Notice how simple Parquet is: **Schema is preserved automatically inside the binary metadata**!
You don't need to specify schemas when reading Parquet.


In [5]:
parquet_path = os.path.join(tmp_dir, "parquet_output")

# Write to Parquet (defaults to Snappy compression)
df.write.mode("overwrite").parquet(parquet_path)

# Read from Parquet
read_parquet_df = spark.read.parquet(parquet_path)
print("Read back from Parquet (Schema automatically preserved!):")
read_parquet_df.printSchema()
read_parquet_df.show(3)


26/11/11 18:56:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Read back from Parquet (Schema automatically preserved!):
root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- year: integer (nullable = true)

+--------+--------+-------+-----------+------+----+
|order_id|customer|country|   category|amount|year|
+--------+--------+-------+-----------+------+----+
|       3| Charlie|     US|Electronics| 120.0|2023|
|       1|   Alice|     US|Electronics|999.99|2023|
|       6|   Fiona|     UK|Electronics| 450.0|2023|
+--------+--------+-------+-----------+------+----+
only showing top 3 rows


---
## 7. Understanding SaveModes

When writing data to a folder that already exists:

| Mode | Behavior |
| :--- | :--- |
| `mode("overwrite")` | Deletes existing directory contents and writes fresh data. |
| `mode("append")` | Adds new files alongside existing files in the directory. |
| `mode("ignore")` | Does nothing if the directory already exists. No error thrown. |
| `mode("errorIfExists")` | **(Default)** Throws an AnalysisException if the path already exists. |


---
## 8. Data Partitioning on Disk (`partitionBy`)

### What is Partitioning on Disk?
Instead of dumping all data into a single huge folder, `partitionBy("country", "year")` organizes files into subdirectories:
```
partitioned_orders/
  ├── country=US/
  │    ├── year=2022/part-0000.parquet
  │    └── year=2023/part-0000.parquet
  ├── country=UK/
  │    └── year=2023/part-0000.parquet
  └── country=CA/
       ├── year=2022/part-0000.parquet
       └── year=2023/part-0000.parquet
```

### Partition Pruning (Massive Speedup):
When you run `SELECT * WHERE country = 'US'`, Spark **never reads the UK or CA folders**! It jumps directly into `country=US`. This is called **Partition Pruning**.


In [6]:
partitioned_path = os.path.join(tmp_dir, "partitioned_orders")

# Write partitioned by country and year
df.write.mode("overwrite").partitionBy("country", "year").parquet(partitioned_path)

print("Directory structure created on disk:")
for root, dirs, files in os.walk(partitioned_path):
    level = root.replace(partitioned_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if f.endswith(".parquet"):
            print(f"{subindent}{f}")


Directory structure created on disk:
partitioned_orders/
    country=CA/
        year=2022/
            part-00003-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
        year=2023/
            part-00006-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
    country=UK/
        year=2023/
            part-00001-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
            part-00005-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
    country=US/
        year=2022/
            part-00004-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
            part-00007-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
        year=2023/
            part-00000-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet
            part-00002-d0b06fa6-a155-4be7-a18c-5f9b6d22fb62.c000.snappy.parquet


In [7]:
# Querying partitioned data:
partitioned_df = spark.read.parquet(partitioned_path)

# Filter on partitioned column: Spark uses Partition Pruning!
us_orders = partitioned_df.filter(partitioned_df.country == "US")
us_orders.show()


+--------+--------+-----------+------+-------+----+
|order_id|customer|   category|amount|country|year|
+--------+--------+-----------+------+-------+----+
|       3| Charlie|Electronics| 120.0|     US|2023|
|       1|   Alice|Electronics|999.99|     US|2023|
|       5|    Evan|   Clothing|  75.0|     US|2022|
|       8|  Hannah|      Books|  35.0|     US|2022|
+--------+--------+-----------+------+-------+----+



---
## 9. ⚠️ The "Small Files Problem" Warning

* **Do NOT partition by high-cardinality columns** (columns with thousands/millions of unique values like `user_id`, `timestamp`, or `email`).
* If you partition by `user_id` with 1 million users, Spark creates 1 million directories and 1 million tiny files!
* File systems and Spark slow to a crawl when opening millions of tiny files.
* **Best practice**: Partition by low-cardinality columns (e.g. `year`, `month`, `country`, `region`).


In [8]:
# Cleanup temporary demo files
if os.path.exists(tmp_dir):
    shutil.rmtree(tmp_dir)
    print("Cleaned up temporary test directory.")


Cleaned up temporary test directory.


---
## 10. Summary of Notebook 08
* Parquet is the industry standard for analytics because of columnar storage, compression, and predicate pushdown.
* Control write behavior using `mode("overwrite")`, `mode("append")`, etc.
* Use `partitionBy()` on low-cardinality columns (`country`, `year`) to enable partition pruning.
* Avoid the small files problem by never partitioning on unique ID columns.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [9]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")


SparkSession stopped successfully! RAM released.
